In [0]:

from pyspark.sql.functions import col, count

# Read Silver table
df = spark.table("workspace.default.silver_retail_sales")

# Run data quality checks
total_records = df.count()

checks = {
    "Missing Order_ID": df.filter(col("Order_ID").isNull()).count(),
    "Missing Customer_ID": df.filter(col("Customer_ID").isNull()).count(),
    "Missing Quantity": df.filter(col("Quantity").isNull()).count(),
    "Invalid Quantity": df.filter(col("Quantity") <= 0).count(),
    "Missing Price": df.filter(col("Price").isNull()).count(),
    "Invalid Price": df.filter(col("Price") <= 0).count(),
    "Negative Revenue": df.filter(col("Revenue") < 0).count(),
    "Duplicate Order_ID": total_records - df.select("Order_ID").distinct().count()
}

print("Total Records:", total_records)
print("\nDATA QUALITY RESULTS")

for check, failures in checks.items():
    status = "PASS" if failures == 0 else "FAIL"
    print(f"{check}: {status} | Failed Records: {failures}")



# Fail the notebook if any data quality check fails
failed_checks = {
    check: failures
    for check, failures in checks.items()
    if failures > 0
}

if failed_checks:
    raise Exception(f"Data Quality Failed: {failed_checks}")

print("All data quality checks passed. Pipeline can continue.")



Total Records: 7

DATA QUALITY RESULTS
Missing Order_ID: PASS | Failed Records: 0
Missing Customer_ID: PASS | Failed Records: 0
Missing Quantity: PASS | Failed Records: 0
Invalid Quantity: PASS | Failed Records: 0
Missing Price: PASS | Failed Records: 0
Invalid Price: PASS | Failed Records: 0
Negative Revenue: PASS | Failed Records: 0
Duplicate Order_ID: PASS | Failed Records: 0
All data quality checks passed. Pipeline can continue.
